# Notebook 13 — The Diver in the Loop

**Companion to Chapter 13**

This notebook treats a diver as part of a delayed, intermittent feedback loop. It compares an ideal continuous benchmark with deadband, sampled decisions, command holding, saturation, and a simple predictive rule.

> **Scope.** This is an engineering model for exploring feedback mechanisms. It is not a physiological model, a training standard, or advice for real diving.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

m = 85.0                 # kg
rho = 1025.0             # kg/m^3
g = 9.80665              # m/s^2
z_star = 20.0            # m, positive downward
V_g0 = 8.0e-3            # m^3 at the surface
p_atm = 101325.0          # Pa
p_star = p_atm + rho*g*z_star
V_g_star = V_g0*p_atm/p_star
k_B = -rho**2*g**2*V_g_star/p_star
a = k_B/m

A = np.array([[0.0, -1.0], [a, 0.0]])
B = np.array([[0.0], [1.0/m]])         # positive input force is upward
C = np.array([[1.0, 0.0]])
D = np.zeros((1, 1))

print(f"Local buoyancy slope k_B = {k_B:.4f} N/m")
print(f"Plant coefficient a = {a:.6f} s^-2")

## 1. Human-controller model

The simulation isolates four mechanisms:

- **deadband:** small perceived depth errors trigger no proportional correction;
- **delay:** decisions use an earlier state;
- **intermittency:** commands are updated only at decision instants;
- **saturation:** available corrective force is bounded.

Between decisions, the last command is held.

In [ ]:
K_p, K_d = 70.0, 140.0

def deadband(x, width):
    return np.sign(x)*max(abs(x)-width,0.0)

def simulate_human(delay=0.6, decision_interval=0.5, width=0.05,
                   predictive=False, T_predict=0.6, duration=35, dt=0.01,
                   u_max=80.0, x0=(0.5,0.0)):
    t = np.arange(0,duration+dt,dt)
    x = np.zeros((2,len(t))); x[:,0]=x0
    u = np.zeros(len(t)); actions=np.zeros(len(t),dtype=bool)
    held=0.0; next_decision=0.0
    for k in range(len(t)-1):
        if t[k]+1e-12 >= next_decision:
            j=max(0,k-int(round(delay/dt)))
            z_seen,v_seen=x[:,j]
            z_for_rule=z_seen-v_seen*T_predict if predictive else z_seen
            held=np.clip(K_p*deadband(z_for_rule,width)-K_d*v_seen,-u_max,u_max)
            actions[k]=True; next_decision += decision_interval
        u[k]=held
        # RK4 for the linear plant with held input
        f=lambda state: A@state+B[:,0]*held
        s=x[:,k]; h=dt
        k1=f(s); k2=f(s+h*k1/2); k3=f(s+h*k2/2); k4=f(s+h*k3)
        x[:,k+1]=s+h*(k1+2*k2+2*k3+k4)/6
    u[-1]=held
    return t,x,u,actions

def simulate_continuous(duration=35,dt=0.01,x0=(0.5,0.0),u_max=80):
    t=np.arange(0,duration+dt,dt)
    def rhs(t,x):
        uu=np.clip(K_p*x[0]-K_d*x[1],-u_max,u_max)
        return A@x+B[:,0]*uu
    sol=solve_ivp(rhs,(0,duration),x0,t_eval=t,rtol=1e-9,atol=1e-11)
    u=np.clip(K_p*sol.y[0]-K_d*sol.y[1],-u_max,u_max)
    return t,sol.y,u

## 2. Continuous benchmark versus intermittent response

In [ ]:
t,x_c,u_c=simulate_continuous()
t,x_h,u_h,actions=simulate_human()

fig,axes=plt.subplots(3,1,sharex=True,figsize=(8,8))
axes[0].plot(t,x_c[0],label="continuous benchmark")
axes[0].plot(t,x_h[0],label="delayed intermittent")
axes[0].set_ylabel("Depth error [m]"); axes[0].legend()
axes[1].plot(t,x_h[1]); axes[1].set_ylabel("Upward velocity [m/s]")
axes[2].step(t,u_h,where="post"); axes[2].plot(t[actions],u_h[actions],"o",ms=3)
axes[2].set(xlabel="Time [s]",ylabel="Held force [N]")
plt.show()

## 3. Delay and decision rate are distinct

Delay makes each decision older; a longer decision interval holds each command for longer. Both can degrade regulation, but they are not interchangeable.

In [ ]:
fig,axes=plt.subplots(2,1,sharex=True,figsize=(8,7))
for delay in [0.0,0.4,0.8]:
    tt,xx,uu,aa=simulate_human(delay=delay)
    axes[0].plot(tt,xx[0],label=f"delay {delay:.1f} s")
for interval in [0.2,0.5,1.0]:
    tt,xx,uu,aa=simulate_human(decision_interval=interval)
    axes[1].plot(tt,xx[0],label=f"decision interval {interval:.1f} s")
axes[0].set_ylabel("Depth error [m]"); axes[1].set_ylabel("Depth error [m]")
axes[1].set_xlabel("Time [s]"); axes[0].legend(); axes[1].legend(); plt.show()

## 4. A simple predictive rule

With upward-positive velocity, $z(t+T)\approx z(t)-v(t)T$. Using this predicted depth can partially compensate delay, but prediction errors can also make performance worse.

In [ ]:
t,x_r,u_r,a_r=simulate_human(predictive=False)
t,x_p,u_p,a_p=simulate_human(predictive=True,T_predict=0.6)
fig,ax=plt.subplots()
ax.plot(t,x_r[0],label="reactive")
ax.plot(t,x_p[0],label="predictive")
ax.set(xlabel="Time [s]",ylabel="Depth error [m]",title="Reactive and predictive intermittent rules")
ax.legend(); plt.show()

## 5. Compare several performance dimensions

In [ ]:
def human_metrics(t,x,u,actions):
    return {"RMS depth error [m]":np.sqrt(np.mean(x[0]**2)),
            "maximum |error| [m]":np.max(np.abs(x[0])),
            "RMS force [N]":np.sqrt(np.mean(u**2)),
            "number of decisions":int(actions.sum())}

print("Reactive:",human_metrics(t,x_r,u_r,a_r))
print("Predictive:",human_metrics(t,x_p,u_p,a_p))
assert np.isfinite(x_p).all()

## Engineering exercises

1. Sweep delay and decision interval on a grid; map RMS depth error.
2. Vary deadband width. Identify the tradeoff between action frequency and residual motion.
3. Add biased depth perception and test whether integral-like adaptation is needed.
4. Change $T_{predict}$ and show that excessive prediction can be harmful.
5. Replace the local linear plant with the nonlinear buoyancy model from Part II.

## Summary

The diver-in-the-loop model is a hybrid dynamical system: continuous motion alternates with discrete, delayed decisions. Deadband can reduce unnecessary activity, while delay, sparse decisions, and saturation can create oscillation or poor recovery. The results are mechanism studies—not claims about individual skill or operational safety.